In [ ]:
import pandas as pd
import re

def clean_and_explode_excel(input_file, output_file):
    all_sheets = pd.read_excel(input_file, sheet_name=None, engine='openpyxl')
    processed_sheets = {}

    prefix_rules = {
        'TREATMENT': r'^TREATMENT:\s*',
        'FACTOR': r'^FACTOR:\s*',
        'COEXISTS': r'^COEXISTS:\s*'
    }

    for sheet_name, df in all_sheets.items():
        print(f"正在处理工作表: {sheet_name} ...")
        if df.shape[1] < 4: 
            print(f"警告: {sheet_name} 列数不足，跳过...")
            processed_sheets[sheet_name] = df
            continue
        target_col = df.iloc[:, 3].astype(str)
        cleaned_series = target_col.copy()
        rule_applied = False
        for key, pattern in prefix_rules.items():
            if key in sheet_name.upper(): 
                cleaned_series = target_col.str.replace(pattern, '', regex=True)
                rule_applied = True
                print(f"  应用规则: 删除 '{key}' 前缀")
                break
        if not rule_applied:
            print(f"  警告: 未找到 {sheet_name} 的匹配规则，跳过前缀删除。")
        temp_df = df.copy()
        temp_df['Cleaned_Entity'] = cleaned_series
        new_rows = []
        for idx, row in temp_df.iterrows():
            entity_str = row['Cleaned_Entity']
            if entity_str == 'Not found' or ',' not in entity_str:
                new_row = row.copy()
                new_row[df.columns[3]] = entity_str 
                new_rows.append(new_row)
            else:
                entities = [e.strip() for e in entity_str.split(',') if e.strip()]
                for entity in entities:
                    new_row = row.copy()
                    new_row[df.columns[3]] = entity 
                    new_rows.append(new_row)
        
        exploded_df = pd.DataFrame(new_rows).reset_index(drop=True)
        
        if 'Cleaned_Entity' in exploded_df.columns:
            exploded_df.drop(columns=['Cleaned_Entity'], inplace=True)
        
        print(f"  处理完成。原始行数: {len(df)}, 新行数: {len(exploded_df)}")
        
        processed_sheets[sheet_name] = exploded_df

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        for name, sheet_df in processed_sheets.items():
            sheet_df.to_excel(writer, sheet_name=name, index=False)
    
    print(f"\n所有工作表处理完成！已保存为: {output_file}")

if __name__ == "__main__":
    input_path = "./copd_answers_chunked.xlsx" # 输入文件名
    output_path = "./LLM回答分词版.xlsx" # 输出文件名

    try:
        clean_and_explode_excel(input_path, output_path)
    except FileNotFoundError:
        print(f"错误: 找不到文件 '{input_path}'。请确保文件名正确且与脚本在同一目录下。")
    except Exception as e:
        print(f"处理过程中发生错误: {e}")

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
import os
from difflib import SequenceMatcher
import networkx as nx

def str_sim(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def share_token(a, b):
    stop_words = {
        'inhaler', 'inhalation', 'nebulizer', 'neb', 'solution', 'soln',
        'tablet', 'tab', 'capsule', 'cap', 'pill', 'medicine', 'med',
        'disease', 'disorder', 'syndrome', 'condition', 'illness',
        'chronic', 'acute', 'severe', 'mild', 'moderate',
        'history', 'hx', 'of', 'and', 'or', 'the', 'a', 'an',
        'mg', 'ml', 'g', 'kg', 'dose', 'dosage', 'pill', 'pill',
        'treatment', 'therapy', 'management', 'care',
        'patient', 'case', 'report', 'study',

        'status', 'post', 's/p', 'secondary', 'due', 'related', 'induced',
        'positive', 'negative', 'old', 'etoh', 'alcoholic', 'non', 'with', 'without',

        'failure', 'insufficiency', 'injury', 'damage', 'lesion', 'tumor', 
        'lesions', 'tumors', 'cancer', 'carcinoma', 'adenocarcinoma', 'neoplasm',

        'system', 'vascular', 'artery', 'arteries', 'vein', 'vessel', 'dysfunction'
    }
    
    a_tokens = set(a.lower().split())
    b_tokens = set(b.lower().split())
    
    a_filtered = a_tokens - stop_words
    b_filtered = b_tokens - stop_words
    
    return len(a_filtered & b_filtered) > 0

def normalize_with_sentencet5(input_file, output_file, model_path, target_col_index=3):
    print(f"正在加载本地模型: {model_path} ...")
    
    if not os.path.exists(model_path):
        print(f"错误：找不到文件夹 {model_path}，请检查路径是否正确！")
        return

    model = SentenceTransformer(model_path)
    
    try:
        all_sheets = pd.read_excel(input_file, sheet_name=None)
    except Exception as e:
        print(f"读取Excel文件失败: {e}")
        return
    
    print(f"发现 {len(all_sheets)} 个工作表，开始处理...")

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        
        for sheet_name, df in all_sheets.items():
            print(f"\n--- 处理工作表: {sheet_name} ---")
            
            entity_col = df.columns[target_col_index]
            entities = df[entity_col].astype(str).tolist()
            
            valid_entities = [
                e for e in entities
                if e.lower() not in ['not found', 'nan', '', 'none'] and len(e) >= 2
            ]
            
            if not valid_entities:
                print("  无有效实体，跳过。")
                df.to_excel(writer, sheet_name=sheet_name, index=False)
                continue

            freq_map = pd.Series(valid_entities).value_counts().to_dict()
            unique_entities = list(freq_map.keys())
            
            print(f"  发现 {len(unique_entities)} 个唯一实体，正在计算语义向量...")
            
            embeddings = model.encode(unique_entities, convert_to_tensor=True, show_progress_bar=True)
            
            cosine_scores = util.cos_sim(embeddings, embeddings)

            threshold = 0.93
            str_threshold = 0.6

            G = nx.Graph()

            for i in range(len(unique_entities)):
                for j in range(i + 1, len(unique_entities)):
                    
                    sim_score = cosine_scores[i][j].item()
                    str_score = str_sim(unique_entities[i], unique_entities[j])
                    token_ok = share_token(unique_entities[i], unique_entities[j])

                    if (
                        sim_score >= threshold
                        and str_score >= str_threshold
                        and token_ok
                    ):
                        G.add_edge(unique_entities[i], unique_entities[j])

            clusters = list(nx.connected_components(G))

            replacement_rules = {}

            def choose_standard(cluster):
                cluster_list = list(cluster)
                short_candidates = sorted(cluster_list, key=len)[:3]
                
                best_candidate = max(short_candidates, key=freq_map.get)
                
                if freq_map.get(best_candidate, 0) < 2:
                     return min(cluster_list, key=len)
                
                return best_candidate

            for cluster in clusters:
                if len(cluster) > 1:
                    standard_name = choose_standard(cluster)
                    
                    full_cluster_list = sorted(list(cluster))
                    
                    print(f"  [聚类] 成员数: {len(cluster)} -> 标准词: '{standard_name}'")
                    print(f"    包含成员: {full_cluster_list}")

                    for word in cluster:
                        if word != standard_name:
                            replacement_rules[word] = standard_name

            if replacement_rules:
                print(f"  [合并] 共替换 {len(replacement_rules)} 个变体")
                df[entity_col] = df[entity_col].replace(replacement_rules)
            else:
                print("  [合并] 未发现高相似度变体。")

            df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"\n所有工作表处理完毕！文件已保存为: {output_file}")

if __name__ == "__main__":
    input_path = "./LLM回答分词版.xlsx"
    output_path = "./LLM回答归一化处理.xlsx"
    
    local_model_path = r"sentence-t5-base"
    
    normalize_with_sentencet5(input_path, output_path, local_model_path)

In [ ]:
import pandas as pd

input_file = "./LLM回答归一化处理.xlsx" 
output_file = "./LLM回答实体频数.xlsx" 
sheet_names = ["TREATMENT","FACTOR","COEXISTS"] 

all_results = {}

print("正在处理数据...")

for sheet_name in sheet_names:
    try:
        df = pd.read_excel(input_file, sheet_name=sheet_name, header=None)
        
        column_d = df.iloc[:, 3] 
        
        cleaned_data = column_d.dropna().astype(str).str.strip() 
        filtered_data = cleaned_data[
            ~cleaned_data.isin(['Not found', 'nan', '']) & 
            (cleaned_data.str.len() > 0)
        ]
        
        freq_count = filtered_data.value_counts()
        
        result_df = freq_count.reset_index()
        result_df.columns = ['Entity', 'Frequency'] 
        
        all_results[sheet_name] = result_df
        
        print(f"✅ {sheet_name}: 找到 {len(result_df)} 个唯一实体")
        
    except FileNotFoundError:
        print(f"❌ 错误：找不到文件 {input_file}，请检查文件名是否正确！")
        exit()
    except Exception as e:
        print(f"❌ 处理工作表 {sheet_name} 时发生错误: {e}")
        continue

if all_results:
    try:
        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
            for sheet_name, data in all_results.items():
                data.to_excel(writer, sheet_name=sheet_name, index=False)
        
        print(f"\n🎉 处理完成！")
        print(f"📊 结果已保存至: {output_file}")
        
        print(f"\n📋 预览 (TREATMENT 前10条):")
        if "TREATMENT" in all_results:
            print(all_results["TREATMENT"].head(10))
            
    except Exception as e:
        print(f"❌ 保存文件失败: {e}")
else:
    print("❌ 没有数据可写入，请检查输入文件格式。")

In [ ]:
from pyvis.network import Network

net = Network(height="800px", width="100%", directed=True, notebook=True)

color_disease = "#FFA500"      
color_drug = "#90EE90"        
color_class = "#E6E6FA"       
color_factor = "#FFCCCB"      
color_coexist = "#ADD8E6"     

net.add_node("COPD", label="COPD", title="Chronic Obstructive Pulmonary Disease", color=color_disease, shape="dot")

net.add_node("Smoking", label="Smoking", color=color_factor, shape="box")
net.add_node("ETOH_abuse", label="ETOH Abuse", color=color_factor, shape="box")
net.add_node("Asbestos_exposure", label="Asbestos Exposure", color=color_factor, shape="box")
net.add_node("Obesity", label="Obesity", color=color_factor, shape="box")
net.add_node("Asthma", label="Asthma", color=color_factor, shape="box")
net.add_node("Hypertension", label="Hypertension", color=color_factor, shape="box")

net.add_node("Albuterol", label="Albuterol", color=color_drug, shape="box")
net.add_node("Atrovent", label="Atrovent", color=color_drug, shape="box")
net.add_node("Lasix", label="Lasix", color=color_drug, shape="box")
net.add_node("Levofloxacin", label="Levofloxacin", color=color_drug, shape="box")
net.add_node("Vancomycin", label="Vancomycin", color=color_drug, shape="box")
net.add_node("Bilevel Positive Airway Pressure", label="Bilevel Positive Airway Pressure", color=color_drug, shape="box")

net.add_node("Prednisone", label="Prednisone", color=color_drug, shape="box")
net.add_node("Solumedrol", label="Solumedrol", color=color_drug, shape="box")
net.add_node("Steroids", label="Steroids", color=color_drug, shape="box")

net.add_node("Diabetes", label="Diabetes", color=color_coexist, shape="box")
net.add_node("Hyperlipidemia", label="Hyperlipidemia", color=color_coexist, shape="box")
net.add_node("Coronary Artery Disease", label="Coronary Artery Disease", color=color_coexist, shape="box")
net.add_node("Congestive Heart Failure", label="Congestive Heart Failure", color=color_coexist, shape="box")

net.add_edge("Smoking", "COPD", label="Factor", color='red', arrows="to")
net.add_edge("ETOH_abuse", "COPD", label="Factor", color='red', arrows="to")
net.add_edge("Asbestos_exposure", "COPD", label="Factor", color='red', arrows="to")
net.add_edge("Obesity", "COPD", label="Factor", color='red', arrows="to")
net.add_edge("Asthma", "COPD", label="Factor", color='red', arrows="to")
net.add_edge("Hypertension", "COPD", label="Factor", color='red', arrows="to")

net.add_edge("COPD", "Diabetes", label="Coexists", arrows="to", color="blue")
net.add_edge("COPD", "Hyperlipidemia", label="Coexists", arrows="to", color="blue")
net.add_edge("COPD", "Coronary Artery Disease", label="Coexists", arrows="to", color="blue")
net.add_edge("COPD", "Congestive Heart Failure", label="Coexists", arrows="to", color="blue")

net.add_edge("COPD", "Albuterol", label="Treatment", arrows="to")
net.add_edge("COPD", "Atrovent", label="Treatment", arrows="to")
net.add_edge("COPD", "Lasix", label="Treatment", title="Treatment", arrows="to")
net.add_edge("COPD", "Levofloxacin", label="Treatment", arrows="to")
net.add_edge("COPD", "Vancomycin", label="Treatment", arrows="to")
net.add_edge("COPD", "Bilevel Positive Airway Pressure", label="Treatment", arrows="to")

# 直接连接 COPD → Prednisone 和 Solumedrol
net.add_edge("COPD", "Prednisone", label="Treatment", arrows="to")
net.add_edge("COPD", "Solumedrol", label="Treatment", arrows="to")
net.add_edge("COPD", "Steroids", label="Treatment", arrows="to")

output_path = "./copd_graph_v4.html" 
net.show(output_path)

print(f"✅ 图谱已生成！请在浏览器中打开: {output_path}")